#### 랭체인 Tool
* 랭체인의 툴(Tool)은 AI가 실시간 검색, 수학 계산, DB 조회 등 외부 기능과 상호작용할 수 있도록 돕는 '기능적 도구'입니다.
* 똑똑하지만 손발이 없는 AI에게 인터넷 검색창, 계산기, 파일 읽기/쓰기 권한 등의 장비를 쥐여주는 것과 같습니다.
* 각 툴은 이름, 설명, 실행 함수로 구성되며, AI는 '설명'을 읽고 이 도구를 언제 꺼내 써야 할지 스스로 판단합니다.
* 주로 에이전트(Agent)와 함께 결합하여, AI가 상황에 맞는 도구를 골라 복잡한 문제를 주도적으로 해결할 때 필수적입니다.

In [37]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈습니다! 당신은 어떻게 지내시나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 12, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da56f7d23d', 'id': 'chatcmpl-EKun4UpMFas9Gmu2CjseRMinBCae8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0740f-8809-7a63-bc3b-d06467c3a919-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 15, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [38]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [39]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
from rich.pretty import pprint
pprint(messages)

[
│   SystemMessage(
│   │   content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.',
│   │   additional_kwargs={},
│   │   response_metadata={}
│   ),
│   HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
│   AIMessage(
│   │   content='',
│   │   additional_kwargs={'refusal': None},
│   │   response_metadata={
│   │   │   'token_usage': {
│   │   │   │   'completion_tokens': 23,
│   │   │   │   'prompt_tokens': 135,
│   │   │   │   'total_tokens': 158,
│   │   │   │   'completion_tokens_details': {
│   │   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   'rejected_prediction_tokens': 0,
│   │   │   │   │   'text_tokens': None
│   │   │   │   },
│   │   │   │   'prompt_tokens_details': {
│   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   'cache_write_tokens': None,
│   │   │   │   │   'cached_tokens': 0,
│   │   │   │   │   'image_tokens': None,
│   │   │   │   │   'text_tokens': None
│   │   │   │   }
│   │   │   },
│   │   │   'model_provider': 'openai',
│   │   │   'model_name': 'gpt-4o-mini-2024-07-18',
│   │   │   'system_fingerprint': 'fp_f367f9dc44',
│   │   │   'id': 'chatcmpl-EKun5tmCaLJU38K49UXZzML9PYIiE',
│   │   │   'service_tier': 'default',
│   │   │   'finish_reason': 'tool_calls',
│   │   │   'logprobs': None
│   │   },
│   │   id='lc_run--01a0740f-8b52-74c1-bfce-b1e626f7aef0-0',
│   │   tool_calls=[
│   │   │   {
│   │   │   │   'name': 'get_current_time',
│   │   │   │   'args': {'timezone': 'Asia/Seoul', 'location': '부산'},
│   │   │   │   'id': 'call_KkDR5EJbrjvGSy6AolAhgQXt',
│   │   │   │   'type': 'tool_call'
│   │   │   }
│   │   ],
│   │   invalid_tool_calls=[],
│   │   usage_metadata={
│   │   │   'input_tokens': 135,
│   │   │   'output_tokens': 23,
│   │   │   'total_tokens': 158,
│   │   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   │   }
│   )
]

In [40]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

pprint(messages)

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-09-06 09:12:51 


[
│   SystemMessage(
│   │   content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.',
│   │   additional_kwargs={},
│   │   response_metadata={}
│   ),
│   HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
│   AIMessage(
│   │   content='',
│   │   additional_kwargs={'refusal': None},
│   │   response_metadata={
│   │   │   'token_usage': {
│   │   │   │   'completion_tokens': 23,
│   │   │   │   'prompt_tokens': 135,
│   │   │   │   'total_tokens': 158,
│   │   │   │   'completion_tokens_details': {
│   │   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   'reasoning_tokens': 0,
│   │   │   │   │   'rejected_prediction_tokens': 0,
│   │   │   │   │   'text_tokens': None
│   │   │   │   },
│   │   │   │   'prompt_tokens_details': {
│   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   'cache_write_tokens': None,
│   │   │   │   │   'cached_tokens': 0,
│   │   │   │   │   'image_tokens': None,
│   │   │   │   │   'text_tokens': None
│   │   │   │   }
│   │   │   },
│   │   │   'model_provider': 'openai',
│   │   │   'model_name': 'gpt-4o-mini-2024-07-18',
│   │   │   'system_fingerprint': 'fp_f367f9dc44',
│   │   │   'id': 'chatcmpl-EKun5tmCaLJU38K49UXZzML9PYIiE',
│   │   │   'service_tier': 'default',
│   │   │   'finish_reason': 'tool_calls',
│   │   │   'logprobs': None
│   │   },
│   │   id='lc_run--01a0740f-8b52-74c1-bfce-b1e626f7aef0-0',
│   │   tool_calls=[
│   │   │   {
│   │   │   │   'name': 'get_current_time',
│   │   │   │   'args': {'timezone': 'Asia/Seoul', 'location': '부산'},
│   │   │   │   'id': 'call_KkDR5EJbrjvGSy6AolAhgQXt',
│   │   │   │   'type': 'tool_call'
│   │   │   }
│   │   ],
│   │   invalid_tool_calls=[],
│   │   usage_metadata={
│   │   │   'input_tokens': 135,
│   │   │   'output_tokens': 23,
│   │   │   'total_tokens': 158,
│   │   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   │   }
│   ),
│   ToolMessage(
│   │   content='Asia/Seoul (부산) 현재시각 2026-09-06 09:12:51 ',
│   │   name='get_current_time',
│   │   tool_call_id='call_KkDR5EJbrjvGSy6AolAhgQXt'
│   )
]

In [41]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 9월 6일 09시 12분 51초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 192, 'total_tokens': 218, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f367f9dc44', 'id': 'chatcmpl-EKun65rE8Y4hW3y3GvdsETvuDnYCH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0740f-8e1d-75e3-9bad-aab2975a98fd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 26, 'total_tokens': 218, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

#### 파이단틱
* 파이단틱(Pydantic)은 파이썬(Python)에서 데이터의 형태(타입)를 검증하고 관리하기 위해 가장 널리 쓰이는 데이터 유효성 검사 라이브러리입니다.
* 클래스 형식으로 데이터 모델을 정의해두면, 들어오는 데이터가 올바른 형식인지 자동으로 체크하고 지정한 타입으로 변환해 줍니다.
* 랭체인에서는 AI가 단순 텍스트가 아닌 "이름", "가격" 등 구조화된 JSON 데이터로 정확히 답변하도록 출력 형식을 강제할 때 필수적으로 사용됩니다.
* 데이터가 잘못 들어오면 명확한 에러를 발생시켜 프로그램의 안정성을 높여주며, FastAPI 등 현대적인 파이썬 웹 프레임워크의 핵심 기반이기도 합니다.

In [42]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [43]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [44]:
messages.append(HumanMessage("팔란티어는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
pprint(response)
messages.append(response)

AIMessage(
│   content='',
│   additional_kwargs={'refusal': None},
│   response_metadata={
│   │   'token_usage': {
│   │   │   'completion_tokens': 27,
│   │   │   'prompt_tokens': 285,
│   │   │   'total_tokens': 312,
│   │   │   'completion_tokens_details': {
│   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   'audio_tokens': 0,
│   │   │   │   'reasoning_tokens': 0,
│   │   │   │   'rejected_prediction_tokens': 0,
│   │   │   │   'text_tokens': None
│   │   │   },
│   │   │   'prompt_tokens_details': {
│   │   │   │   'audio_tokens': 0,
│   │   │   │   'cache_write_tokens': None,
│   │   │   │   'cached_tokens': 0,
│   │   │   │   'image_tokens': None,
│   │   │   │   'text_tokens': None
│   │   │   }
│   │   },
│   │   'model_provider': 'openai',
│   │   'model_name': 'gpt-4o-mini-2024-07-18',
│   │   'system_fingerprint': 'fp_af8966b59d',
│   │   'id': 'chatcmpl-EKun7ZSPr4EuKjnUFpfV7lJ7fDb2i',
│   │   'service_tier': 'default',
│   │   'finish_reason': 'tool_calls',
│   │   'logprobs': None
│   },
│   id='lc_run--01a0740f-93bb-7200-bdfc-d36104ef2c7b-0',
│   tool_calls=[
│   │   {
│   │   │   'name': 'get_yf_stock_history',
│   │   │   'args': {'stock_history_input': {'ticker': 'PLTR', 'period': '1mo'}},
│   │   │   'id': 'call_Hp9oGK75ZJGATtLAt6t5T40d',
│   │   │   'type': 'tool_call'
│   │   }
│   ],
│   invalid_tool_calls=[],
│   usage_metadata={
│   │   'input_tokens': 285,
│   │   'output_tokens': 27,
│   │   'total_tokens': 312,
│   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   }
)

In [45]:
from IPython.display import Markdown, display

for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    display(Markdown(str(tool_msg.content)))

{'stock_history_input': {'ticker': 'PLTR', 'period': '1mo'}}


| Date                      |    Open |    High |     Low |   Close |      Volume |   Dividends |   Stock Splits |
|:--------------------------|--------:|--------:|--------:|--------:|------------:|------------:|---------------:|
| 2026-08-05 00:00:00-04:00 | 162     | 166.08  | 158.25  |  158.43 | 6.35159e+07 |           0 |              0 |
| 2026-08-06 00:00:00-04:00 | 155.87  | 158     | 152.7   |  155.92 | 4.17514e+07 |           0 |              0 |
| 2026-08-07 00:00:00-04:00 | 160.07  | 172.41  | 159.96  |  172.01 | 7.75799e+07 |           0 |              0 |
| 2026-08-10 00:00:00-04:00 | 171.01  | 179.6   | 170.87  |  175.23 | 5.74027e+07 |           0 |              0 |
| 2026-08-11 00:00:00-04:00 | 174.18  | 177.94  | 172.72  |  174.94 | 4.31625e+07 |           0 |              0 |
| 2026-08-12 00:00:00-04:00 | 174.07  | 175.09  | 168.34  |  171.04 | 3.52203e+07 |           0 |              0 |
| 2026-08-13 00:00:00-04:00 | 173.4   | 179.91  | 172.33  |  179.01 | 3.65112e+07 |           0 |              0 |
| 2026-08-14 00:00:00-04:00 | 179.48  | 180.18  | 173.8   |  174.04 | 2.40658e+07 |           0 |              0 |
| 2026-08-17 00:00:00-04:00 | 174.06  | 176.275 | 172.301 |  172.55 | 2.45909e+07 |           0 |              0 |
| 2026-08-18 00:00:00-04:00 | 171.97  | 174.98  | 170.61  |  171.54 | 2.80821e+07 |           0 |              0 |
| 2026-08-19 00:00:00-04:00 | 172.61  | 176.82  | 169.771 |  175.19 | 3.61326e+07 |           0 |              0 |
| 2026-08-20 00:00:00-04:00 | 175.88  | 176.46  | 172.03  |  173.96 | 2.70184e+07 |           0 |              0 |
| 2026-08-21 00:00:00-04:00 | 173.98  | 182.44  | 172.55  |  179.94 | 4.10762e+07 |           0 |              0 |
| 2026-08-24 00:00:00-04:00 | 177.39  | 178.75  | 171.31  |  175.89 | 3.51155e+07 |           0 |              0 |
| 2026-08-25 00:00:00-04:00 | 176.63  | 179.87  | 172.2   |  172.73 | 2.45585e+07 |           0 |              0 |
| 2026-08-26 00:00:00-04:00 | 170.6   | 178.49  | 168.9   |  177.5  | 2.79242e+07 |           0 |              0 |
| 2026-08-27 00:00:00-04:00 | 178.75  | 186.86  | 178.01  |  185.93 | 4.07393e+07 |           0 |              0 |
| 2026-08-28 00:00:00-04:00 | 184.95  | 188.37  | 184.38  |  186.29 | 2.50602e+07 |           0 |              0 |
| 2026-08-31 00:00:00-04:00 | 184.495 | 187.94  | 183.79  |  186.38 | 2.59412e+07 |           0 |              0 |
| 2026-09-01 00:00:00-04:00 | 182.98  | 186.55  | 179.75  |  179.92 | 2.45672e+07 |           0 |              0 |
| 2026-09-02 00:00:00-04:00 | 176.99  | 177.57  | 165.71  |  169.46 | 3.9874e+07  |           0 |              0 |
| 2026-09-03 00:00:00-04:00 | 172.5   | 185.83  | 171.91  |  182.53 | 3.8108e+07  |           0 |              0 |
| 2026-09-04 00:00:00-04:00 | 180.01  | 182.19  | 173.67  |  174.33 | 2.78953e+07 |           0 |              0 |

In [46]:
llm_with_tools.invoke(messages)

AIMessage(content='팔란티어(PLTR)의 주가는 한 달 전(2026년 8월 5일)에는 158.43 달러로 마감했으며, 현재(2026년 9월 4일)에는 174.33 달러로 마감했습니다. \n\n결론적으로, 팔란티어의 주가는 한 달 전에 비해 올랐습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 1753, 'total_tokens': 1837, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_af8966b59d', 'id': 'chatcmpl-EKun8WouH5MklT2xNxVEsBHGgrArn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0740f-96f0-76d0-afd7-faea3ddb1322-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1753, 'output_tokens': 84, 'total_tokens': 1837, 'input_toke